# 🖥️ Local AI Developer Assistant
**Day 10 · AI Application Development Bootcamp**

This lab builds a developer assistant that runs entirely on **your own machine** using Ollama — no cloud API, no internet needed after setup.

---

### How to use this notebook
- Run cells **in order** — later cells depend on earlier ones
- Cells marked `# ✏️ YOUR TURN` have gaps for you to fill in
- Write your reflection in the **📝 Reflection** section at the end

### Sections at a glance

| Part | Topic | Type |
|------|-------|------|
| A | Connect to Ollama | Run & read |
| B | Core generation function | Fill in |
| C | Prompt templates for developer tasks | Fill in |
| D | Multi-turn conversation with `/api/chat` | Fill in |
| E | Task classifier and router | Fill in |
| F | Multi-model benchmark | Fill in |
| G | Optional UI (Streamlit or Gradio) | Open |
| H | Reflection | Write |

### Before you start
Make sure Ollama is running and at least one model is pulled (see handout Section 6):
```bash
ollama serve
```
Suggested models — pick what fits your hardware:

| Model | Min RAM | Notes |
|-------|---------|-------|
| `qwen2.5:1.5b` | 4 GB | Today's default — works on any laptop |
| `llama3.2:1b` | 4 GB | Alternative — equally valid |
| `llama3.2:3b` | 6 GB | Better quality, still fast |
| `qwen3:4b` | 8 GB | Noticeably better if hardware allows |


---
## ⚙️ Setup — Imports


In [1]:
# If needed: !pip install requests pandas

import requests
import time
import pandas as pd
from typing import Dict, Any, List

OLLAMA_URL = "http://localhost:11434"

# ✏️ Change this to whichever model you pulled.
# The benchmark (Part F) will compare this against a second model.
MODEL_NAME = "qwen2.5:1.5b"

print(f"Using model: {MODEL_NAME}")
print(f"Ollama URL : {OLLAMA_URL}")


Using model: qwen2.5:1.5b
Ollama URL : http://localhost:11434


---
## Part A — Connect to Ollama

Before writing any prompts, confirm that Ollama is running and the right models are installed.
If this cell fails, run `ollama serve` in a separate terminal and try again.


In [2]:
def check_ollama() -> bool:
    """Return True if Ollama is reachable and list installed models."""
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        response.raise_for_status()
        models = response.json().get("models", [])
        print("✅ Ollama is running.")
        print(f"   Installed models ({len(models)}):")
        for m in models:
            print("   -", m.get("name"))
        return True
    except Exception as e:
        print("❌ Could not connect to Ollama.")
        print("   Run `ollama serve` in a separate terminal, then try again.")
        print("   Error:", e)
        return False

check_ollama()


✅ Ollama is running.
   Installed models (9):
   - qwen2.5:0.5b
   - nomic-embed-text:latest
   - qwen2.5:1.5b
   - qwen3:4b
   - deepseek-r1:7b
   - qwen3:0.6b
   - llama3:latest
   - gemma3:4b
   - gemma4:e2b


True

---
## Part B — Core Generation Function

This is the **only place in the whole lab where a network call is made**. Every prompt template, classifier, and benchmark you build in later sections calls this one function.

It uses `/api/generate` — one prompt in, one response out, no memory of previous turns. You will add multi-turn memory in Part D.

**Why `temperature=0.2`?**  
Low temperature means the model picks the most likely next token rather than sampling creatively. For developer tasks — debugging, explanation, test generation — you want **consistent, factual answers**, not creative variation. Try higher values (0.7+) only for open-ended tasks.

**What you should see after filling this in:**  
A short direct answer to the test prompt, printed cleanly.


In [3]:
def ask_ollama(prompt: str, model: str = MODEL_NAME, temperature: float = 0.2) -> str:
    """
    Send a single prompt to a local Ollama model and return the response text.

    Endpoint: POST /api/generate
    Request body: {model, prompt, stream, options: {temperature}}
    Response: response.json()["response"]
    """
    url = f"{OLLAMA_URL}/api/generate"

    # ✏️ YOUR TURN: build the payload dict
    # Keys needed: "model", "prompt", "stream" (set to False), "options" ({"temperature": temperature})
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }

    # ✏️ YOUR TURN: send the POST request, raise on error, return the "response" field
    # Hint: requests.post(url, json=payload, timeout=120)
    #       response.raise_for_status()
    #       return response.json().get("response", "").strip()
    request_response = requests.post(url, json=payload, timeout=120)
    request_response.raise_for_status()
    return request_response.json().get("response", "").strip()

    # raise NotImplementedError("Complete ask_ollama() first.")


# ── Smoke test — uncomment after implementing ─────────────────────────────────
answer = ask_ollama("In one sentence: what is a Python function?")
print("✅", answer)


✅ A Python function is a block of code that performs a specific task and can be reused throughout the program.


---
## Part C — Prompt Templates for Developer Tasks

A prompt template wraps the user's raw input (code, error message, etc.) in a structured instruction that produces consistent, usable output from the model.

**Key principles for developer prompts:**
- Give the model a clear role: `"You are a debugging assistant."`
- Wrap code in a fenced block so the model treats it as code, not prose
- Be explicit about output format — ask for numbered steps, not open-ended paragraphs
- Always ask for a corrected or improved version — makes the output actionable

Implement at least **three** of the four templates below.


In [4]:
def make_code_explainer_prompt(code_snippet: str) -> str:
    """
    ✏️ YOUR TURN: write a prompt that explains code to a beginner.

    Include in your prompt:
    - A role instruction  e.g. "You are a helpful programming tutor."
    - The code inside a fenced ```python block
    - Ask for: what the code does, important lines, any bugs or risks,
      and a corrected version if needed
    """
    # YOUR CODE HERE
    return f"""
    You are a helpful Python programming tutor.

    Explain the following Python code to a beginner:

    ```python
    {code_snippet}
    ```

    Please include:

    1. What the code does
    2. Important lines explained simply
    3. Any bugs or risks
    4. A corrected version if needed
    """.strip()


def make_debug_prompt(error_message: str, code_snippet: str = "") -> str:
    """
    ✏️ YOUR TURN: write a debugging prompt.

    Include: a role instruction, the error message clearly labelled,
    the code in a fenced block (if provided).
    Ask for: likely cause, problem line, simple fix, safer rewritten version.
    """
    # YOUR CODE HERE
    return f"""
    You are a helpful Python debugging assistant.

    Error message:
    {error_message}

    Code:

    ```python
    {code_snippet}
    ```

    Please provide:

    1. The likely cause
    2. The problem line
    3. A simple fix
    4. A safer rewritten version
    """.strip()


def make_testcase_prompt(code_snippet: str) -> str:
    """
    ✏️ YOUR TURN: write a test-case suggestion prompt.

    For each test case, ask for: input, expected output, why it matters.
    Include edge cases: empty input, None, wrong type, boundary values.
    """
    # YOUR CODE HERE
    return f"""
    You are a helpful Python programming tutor.

    Suggest test cases for the following Python code:

    ```python
    {code_snippet}
    ```

    For each test case, provide:

    1. The input
    2. The expected output or expected error
    3. Why the test case matters

    Include relevant edge cases such as:

    - Empty input
    - None
    - Incorrect data types
    - Boundary values

    Keep the test cases clear and beginner-friendly.
    """.strip()


def make_improvement_prompt(code_snippet: str) -> str:
    """
    ✏️ YOUR TURN: write a code review / improvement prompt.

    Ask for: readability issues, missing error handling, performance notes,
    and a cleaner rewritten version with brief explanations of each change.
    """
    # YOUR CODE HERE
    return f"""
    You are a helpful Python code reviewer.

    Review the following Python code for improvements:

    ```python
    {code_snippet}
    ```

    Please provide:

    1. Any readability issues
    2. Any missing validation or error handling
    3. Any performance or maintainability concerns
    4. A cleaner rewritten version of the code
    5. A brief explanation of each important change

    Keep the improved code simple and preserve the original behaviour unless a bug needs to be fixed.
    """.strip()

### C2 — Try your templates on a real example

Uncomment each block after implementing the corresponding template. You should see the model produce a structured response for each task type.


In [5]:
sample_code = '''
def divide_numbers(a, b):
    return a / b

print(divide_numbers(10, 0))
'''

sample_error = "ZeroDivisionError: division by zero"

# ── Explanation ───────────────────────────────────────────────────────────────
print("=== 📖 EXPLANATION ===")
print(ask_ollama(make_code_explainer_prompt(sample_code)))

# ── Debug ─────────────────────────────────────────────────────────────────────
print("\n=== 🐛 DEBUG ===")
print(ask_ollama(make_debug_prompt(sample_error, sample_code)))

# ── Test cases ────────────────────────────────────────────────────────────────
print("\n=== 🧪 TEST CASES ===")
print(ask_ollama(make_testcase_prompt(sample_code)))

# ── Improvement ───────────────────────────────────────────────────────────────
print("\n=== ✨ IMPROVEMENT ===")
print(ask_ollama(make_improvement_prompt(sample_code)))


=== 📖 EXPLANATION ===
Sure! Let's break down this Python code step by step.

### 1. What the Code Does

The code defines a function named `divide_numbers` that takes two arguments, `a` and `b`. The function returns the result of dividing `a` by `b`.

Here’s how it works:
- When you call `divide_numbers(10, 0)`, the function will divide 10 by 0.
- This is where things go wrong because division by zero is undefined in mathematics and results in a runtime error.

### 2. Important Lines Explained Simply

- **`def divide_numbers(a, b):`**: This line defines a new function named `divide_numbers`. It takes two parameters: `a` and `b`.
  
- **`return a / b`:** This line calls the division operation on the values of `a` and `b`, returning their quotient.

### 3. Bugs or Risks

The main issue with this code is that it attempts to divide by zero, which results in a runtime error (division by zero). Here’s why:

- **Undefined Behavior**: When you try to perform division on numbers where one of the

---
## Part D — Multi-turn Conversation with `/api/chat`

So far you have used `/api/generate` — one prompt, one response, no memory of previous turns.

`/api/chat` accepts a `messages` list (same format as OpenAI's API). Each call sends the **full conversation history**, so the model can refer back to earlier messages.

This matters for a developer assistant: a user might say *"explain this function"*, then follow up with *"now add error handling to it"* — and the model needs to know what *"it"* refers to.

**Message format:**
```python
[
    {"role": "system",    "content": "..."},   # optional — sets the model's behaviour
    {"role": "user",      "content": "..."},   # first user message
    {"role": "assistant", "content": "..."},   # model's reply — append this after each turn
    {"role": "user",      "content": "..."},   # next user message
]
```

**What you should see:** Turn 1 explains the function. Turn 2 produces a rewritten version with error handling — and references what was discussed in Turn 1, without you repeating the code.


In [6]:
def chat_with_ollama(
    messages: List[Dict[str, str]],
    model: str = MODEL_NAME,
    temperature: float = 0.2
) -> str:
    """
    Multi-turn chat using /api/chat.
    Returns the model's latest reply as a string.

    Endpoint: POST /api/chat
    Request body: {model, messages, stream: False, options: {temperature}}
    Response: response.json()["message"]["content"]
    """
    # ✏️ YOUR TURN
    # Hint: almost identical to ask_ollama, but:
    #   - use /api/chat instead of /api/generate
    #   - the body key is "messages" (the list), not "prompt"
    #   - the response field is response.json()["message"]["content"]
    api_url = f"{OLLAMA_URL}/api/chat"
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature}
    }

    response = requests.post(api_url, json=payload)
    response.raise_for_status()
    return response.json()["message"]["content"]


# ── Two-turn demo — uncomment after implementing chat_with_ollama() ────────────
conversation = [
    {"role": "system", "content": "You are a helpful Python tutor."},
    {"role": "user",   "content": f"Explain this function:\n```python\n{sample_code}\n```"}
]

reply1 = chat_with_ollama(conversation)
print("Turn 1 — Explanation:")
print(reply1)

# ✏️ YOUR TURN: append reply1 as role="assistant", then add the follow-up question
conversation.append({"role": "assistant", "content": reply1})
conversation.append({"role": "user", "content": "Now rewrite it with proper error handling."})

reply2 = chat_with_ollama(conversation)
print("\nTurn 2 — Rewrite:")
print(reply2)


Turn 1 — Explanation:
The provided Python function `divide_numbers` takes two arguments: `a` and `b`. The function returns the result of dividing `a` by `b`.

Here's a step-by-step breakdown:

1. **Function Definition**: 
   ```python
   def divide_numbers(a, b):
       return a / b
   ```
   - This line defines a new function named `divide_numbers`.
   - It takes two parameters: `a` and `b`.
   - The function body is the single statement that performs the division operation using `/`.

2. **Function Call**:
   ```python
   print(divide_numbers(10, 0))
   ```
   - This line calls the `divide_numbers` function with the arguments `10` and `0`.
   - Since division by zero is undefined in mathematics, this call will result in a runtime error.

3. **Output**:
   ```python
   print(divide_numbers(10, 0))
   ```
   - When you run this code, it will output an error message indicating that the function cannot be divided by zero.

### Explanation of the Error

When `divide_numbers` is called wit

---
## Part E — Task Classifier and Router

A real developer assistant should not ask the user to pick a task from a dropdown. It reads the request, decides what kind of task it is, selects the right prompt template automatically, and responds.

This section builds that in two steps:
1. **`classify_task()`** — reads the user's request and returns a task type string
2. **`developer_assistant()`** — uses the task type to select a prompt template and return the answer

The classifier is keyword-based — `if`/`elif` on lowercased text. This is intentional: you don't need machine learning for well-defined categories with reliable keywords. The important design point is **separating classification from generation** — the classifier never calls the model, and the generator doesn't need to know about classification.

**What you should see:** each test request is routed to a different task type and produces the right kind of response.


In [7]:
def classify_task(user_request: str) -> str:
    """
    Classify a free-text developer request into one of four task types.
    Returns one of: "debug", "test", "improve", "explain"

    ✏️ YOUR TURN:
    - Lowercase the request
    - Check for debug keywords:   error, bug, fix, debug, exception, traceback, fails
    - Check for test keywords:    test, case, assert, pytest, unittest, coverage
    - Check for improve keywords: improve, refactor, clean, optimise, optimize, rewrite, review
    - Default to "explain" if none match

    Hint: use any(k in text for k in ["error", "bug", ...])
    """
    # YOUR CODE HERE
    text = user_request.lower()
    if any(k in text for k in ["error", "bug", "fix", "debug", "exception", "traceback", "fails"]):
        return "debug"
    elif any(k in text for k in ["test", "case", "assert", "pytest", "unittest", "coverage"]):
        return "test"
    elif any(k in text for k in ["improve", "refactor", "clean", "optimise", "optimize", "rewrite", "review"]):
        return "improve"
    else:
        return "explain"


def developer_assistant(
    user_request: str,
    code_snippet: str = "",
    error_message: str = ""
) -> Dict[str, str]:
    """
    Full pipeline: classify request → select prompt template → generate answer.
    Returns {"task": task_type, "answer": model_response}
    so you can see both what was classified and what the model said.

    ✏️ YOUR TURN:
    - Call classify_task(user_request)
    - Select the right make_*_prompt() based on the task type
      (for "debug", pass error_message and code_snippet)
    - Call ask_ollama() with the prompt
    - Return {"task": task, "answer": answer}
    """
    # YOUR CODE HERE
    task = classify_task(user_request)

    if task == "debug":
        prompt = make_debug_prompt(error_message, code_snippet)
    elif task == "test":
        prompt = make_testcase_prompt(code_snippet)
    elif task == "improve":
        prompt = make_improvement_prompt(code_snippet)
    else:
        prompt = make_code_explainer_prompt(code_snippet)

    answer = ask_ollama(prompt)
    return {"task": task, "answer": answer}


# ── Test the router on four requests — uncomment after implementing ────────────
requests_to_test = [
    ("Please debug this",       sample_code, sample_error),
    ("Suggest test cases",      sample_code, ""),
    ("Refactor this function",  sample_code, ""),
    ("What does this do?",      sample_code, ""),
]

for req, code_s, err in requests_to_test:
    result = developer_assistant(req, code_s, err)
    print(f"Request  : {req}")
    print(f"Detected : {result['task']}")
    print(f"Answer   : {result['answer'][:250]}...")
    print()


Request  : Please debug this
Detected : debug
Answer   : ### 1. Likely Cause
The error message "ZeroDivisionError: division by zero" indicates that the function `divide_numbers` is attempting to divide a number by zero, which is mathematically undefined and results in an exception.

### 2. Problem Line
The...

Request  : Suggest test cases
Detected : test
Answer   : Sure, let's create some test cases for the `divide_numbers` function. We'll cover various scenarios to ensure our code behaves correctly under different conditions.

### Test Case 1: Normal Division
**Input:** `a = 10`, `b = 2`
**Expected Output or E...

Request  : Refactor this function
Detected : improve
Answer   : ### 1. Readability Issues

The current function `divide_numbers` is straightforward but lacks readability, especially when it comes to its purpose and parameters.

### 2. Missing Validation or Error Handling

There's no validation for the input value...

Request  : What does this do?
Detected : explain
Answer  

---
## Part F — Multi-Model Benchmark

One of the key questions in local AI is: **how much quality do you give up by using a smaller model, and is the speed gain worth it?**

This section makes that concrete: run the same three prompts through two different model sizes and compare results side by side.

**What to think about while reading the results:**
- Is the quality difference noticeable on the simple task? What about the complex one?
- At which task does the smaller model fall apart first?
- If you were building a real coding assistant, which model would you choose — and why?

**What you should see:** a DataFrame with elapsed times and response lengths per model,
plus two full answers side-by-side for the complex task.


In [8]:
# Set your two models. Pick one size up or down from your primary model.
# The models must both be already pulled — `ollama list` to check.
#
# Example pairs:
#   ("qwen2.5:1.5b",  "qwen2.5:0.5b")   — compare within Qwen 2.5
#   ("llama3.2:3b",   "llama3.2:1b")    — compare within Llama 3.2
#   ("qwen3:4b",      "qwen2.5:1.5b")   — compare across families
MODEL_A = MODEL_NAME          # your primary model
MODEL_B = "qwen2.5:0.5b"     # ✏️ change to a second model you have pulled


def benchmark_prompt(prompt: str, model: str) -> Dict[str, Any]:
    """
    Run one prompt through one model and record timing and output length.

    ✏️ YOUR TURN:
    - Record start time with time.perf_counter()
    - Call ask_ollama(prompt, model=model)
    - Record end time and compute elapsed
    - Return: {"model", "prompt_chars", "response_chars", "elapsed_seconds", "answer"}
    """
    # YOUR CODE HERE
    start_time = time.perf_counter()
    answer = ask_ollama(prompt, model=model)
    end_time = time.perf_counter()
    elapsed_seconds = end_time - start_time
    return {
        "model": model,
        "prompt_chars": len(prompt),
        "response_chars": len(answer),
        "elapsed_seconds": elapsed_seconds,
        "answer": answer
    }


benchmark_prompts = [
    ("Simple",
     "Explain recursion in Python in one short paragraph."),

    ("Debug",
     f"Find the bug in this code and explain why it fails:\n```python\n{sample_code}\n```"),

    ("Complex",
     "Write a Python function that reads a CSV file, filters rows where a given column "
     "exceeds a threshold, and returns the result as a list of dicts. "
     "Include type hints, a docstring, and error handling for missing files."),
]

# ✏️ YOUR TURN: run all prompts through both models and build a comparison DataFrame.
#
results = []
for label, prompt in benchmark_prompts:
    for model in [MODEL_A, MODEL_B]:
        row = benchmark_prompt(prompt, model)
        row["task"] = label
        results.append(row)

df = pd.DataFrame(results)
df[["task", "model", "elapsed_seconds", "response_chars"]]


,task,model,elapsed_seconds,response_chars
0,Simple,qwen2.5:1.5b,3.442516,396
1,Simple,qwen2.5:0.5b,6.384164,1040
2,Debug,qwen2.5:1.5b,6.602117,1266
3,Debug,qwen2.5:0.5b,4.520678,1362
4,Complex,qwen2.5:1.5b,7.015722,1671
5,Complex,qwen2.5:0.5b,4.541184,1619


### F2 — Quality comparison

Read both model answers for the **Complex** task side by side and rate each (1 = poor, 5 = excellent).

| Task | Model A answer | Model B answer | Notes |
|------|---------------|----------------|-------|
| Simple | 5/5 | 1/5 | |
| Debug | 5/5 | 4/5 | |
| Complex | 4/5 | 5/5 | |


In [9]:
# ── Print full answers for the Complex task side by side ─────────────────────
# Uncomment after the benchmark cell runs successfully.

complex_prompt = benchmark_prompts[2][1]

for model in [MODEL_A, MODEL_B]:
    print(f"{'='*60}")
    print(f"  {model}")
    print(f"{'='*60}")
    print(ask_ollama(complex_prompt, model=model))
    print()


  qwen2.5:1.5b
```python
import csv

def filter_rows_by_column(file_path: str, column_name: str, threshold: float) -> list:
    """
    Reads a CSV file, filters rows where the value in a specified column exceeds a given threshold,
    and returns the filtered data as a list of dictionaries.
    
    Parameters:
    - file_path (str): The path to the CSV file.
    - column_name (str): The name of the column to filter by.
    - threshold (float): The threshold value for filtering rows.
    
    Returns:
    - List[dict]: A list of dictionaries, each representing a row that meets the criteria.
    
    Raises:
    - FileNotFoundError: If the specified file does not exist.
    """
    try:
        with open(file_path, mode='r', newline='') as csvfile:
            reader = csv.DictReader(csvfile)
            filtered_rows = [row for row in reader if float(row[column_name]) > threshold]
        
        return filtered_rows
    except FileNotFoundError:
        raise FileNotFoundError(f"The

---
## Part G — Optional: UI with Streamlit or Gradio

If time allows, wrap `developer_assistant()` in a simple web UI. The goal is to confirm the same local model works through a browser interface — not to build a polished product.

Save either snippet as a `.py` file and run it from the terminal.

### Option A — Streamlit

```python
# dev_assistant_app.py
# Run with: streamlit run dev_assistant_app.py

import streamlit as st
# import your functions from this notebook or copy them here

st.title("🖥️ Local AI Developer Assistant")
st.caption("Running on Ollama — no internet needed")

code_input    = st.text_area("Paste your code here", height=200)
error_input   = st.text_input("Error message (optional)")
request_input = st.text_input("What do you want?  e.g. explain this, debug this, suggest tests")

if st.button("Run assistant") and request_input:
    with st.spinner("Thinking locally..."):
        result = developer_assistant(request_input, code_input, error_input)
    st.markdown(f"**Classified as:** `{result['task']}`")
    st.markdown(result["answer"])
```

### Option B — Gradio

```python
# dev_assistant_gradio.py
# Run with: python dev_assistant_gradio.py

import gradio as gr
# import your functions from this notebook or copy them here

def run(request, code, error):
    result = developer_assistant(request, code, error)
    return f"Classified as: {result['task']}\n\n{result['answer']}"

gr.Interface(
    fn=run,
    inputs=[
        gr.Textbox(label="What do you want?"),
        gr.Textbox(label="Code (optional)", lines=8),
        gr.Textbox(label="Error message (optional)"),
    ],
    outputs=gr.Textbox(label="Answer", lines=15),
    title="Local AI Developer Assistant"
).launch()
```


---
## 📝 Reflection

Write **150–250 words** addressing the questions below. Connect your answers to what you actually observed in Parts C – F — not just what the handout says.

1. **Model choice and hardware:** which model(s) did you use, and why that size given your machine?
2. **Benchmark results:** was the speed difference between your two models significant? At which task did quality diverge most?
3. **Real-world relevance:** from the handout's examples — Samsung data breach, air-gapped systems, cost at scale — which is most relevant to a project you could imagine building, and why?
4. **When to use cloud:** when would you still choose Groq or OpenAI over a local model for a developer assistant?


*Your reflection (150–250 words):*

For this lab I went with qwen2.5:1.5b as my main model and qwen2.5:0.5b as the comparison. Picked these two mainly because they're lightweight enough to run locally without eating up my resources, and I wanted to see if the smaller model would actually live up to being "faster."

Results were kind of surprising. The 1.5b model won on speed for the Simple task, but the 0.5b model was actually faster on both Debug and Complex. That said, faster didn't mean better. The clearest gap was on the Complex task, where the 0.5b model's answer looked okay at a glance, but it had hard-coded "column_name" instead of taking the column as an argument, which basically made the function useless outside that one specific case. The 1.5b model was slower to respond, but its solution actually worked properly and didn't need fixing.

The Samsung data breach case stood out to me the most out of all the examples, since a coding assistant is exactly the kind of tool that ends up seeing private source code or internal company data. Running everything locally means none of that ever has to leave the machine, which is a pretty big deal if you're working with sensitive stuff.

That said, I'd still reach for something like Groq or OpenAI when I actually need strong reasoning, solid debugging, or just faster turnaround on harder tasks, especially in situations where privacy isn't the main concern.
